# F1 Tyre Degradation and Pit Strategy (AP in Part B)

This notebook is split into:

- **Part A:** estimate tyre degradation rates, fuel/track gain, and pit-stop loss directly from FastF1 data (without AP fitting)
- **Part B:** use the extracted rates in an AP model to compare and optimize pit strategies

In [ ]:
# If needed, run once:
# !pip install fastf1 scipy pandas numpy matplotlib

import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

In [ ]:
import os
os.makedirs('fastf1_cache', exist_ok=True)
fastf1.Cache.enable_cache('fastf1_cache')

## Part A) Extract degradation rates and pit loss from data

In [ ]:
# ---------- Inputs ----------
YEAR = 2025
GRAND_PRIX = 'Monza'
SESSION = 'R'
DRIVER = 'VER'

COMPOUNDS_TO_ANALYZE = ['SOFT', 'MEDIUM', 'HARD']

In [ ]:
session = fastf1.get_session(YEAR, GRAND_PRIX, SESSION)
session.load()

all_laps = session.laps.pick_drivers(DRIVER).copy()
display(all_laps[['Driver', 'LapNumber', 'Stint', 'Compound', 'LapTime', 'PitInTime', 'PitOutTime']].head())

In [ ]:
def prepare_nonpit_laps(laps_df, group_cols):
    laps = laps_df.copy()
    laps = laps[laps['LapTime'].notna()].copy()
    laps['CompoundU'] = laps['Compound'].str.upper()
    laps['LapTimeSec'] = laps['LapTime'].dt.total_seconds()

    # Remove in/out laps for clean pace deltas
    laps = laps[laps['PitInTime'].isna() & laps['PitOutTime'].isna()].copy()
    laps = laps.pick_quicklaps(threshold=1.07)

    laps = laps.sort_values(group_cols + ['LapNumber']).copy()
    laps['TyreLap'] = laps.groupby(group_cols).cumcount() + 1
    laps['LapDeltaSec'] = laps.groupby(group_cols)['LapTimeSec'].diff()
    return laps


def estimate_fuel_track_gain_from_field(session_laps, frontier_quantile=0.2):
    field_nonpit = prepare_nonpit_laps(session_laps, group_cols=['Driver', 'Stint'])
    frontier = field_nonpit.groupby('LapNumber')['LapTimeSec'].quantile(frontier_quantile).dropna().sort_index()
    deltas = frontier.diff().dropna()
    if len(deltas) < 5:
        raise ValueError('Not enough field-lap deltas to estimate fuel/track gain.')

    q1, q3 = np.quantile(deltas, [0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    deltas = deltas[(deltas >= low) & (deltas <= high)]
    return float(np.median(deltas))


def estimate_driver_deg_rates(driver_nonpit, compounds, fuel_track_gain):
    raw_deg = {}
    used_deg = {}
    observed_delta = {}

    for c in compounds:
        tmp = driver_nonpit[driver_nonpit['CompoundU'] == c].copy()
        deltas = tmp['LapDeltaSec'].dropna()
        if len(deltas) < 4:
            continue

        q1, q3 = np.quantile(deltas, [0.25, 0.75])
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        deltas = deltas[(deltas >= low) & (deltas <= high)]
        if len(deltas) < 3:
            continue

        med_delta = float(np.median(deltas))
        d_raw = med_delta - fuel_track_gain
        d_used = max(d_raw, 0.0)  # physical lower bound

        observed_delta[c] = med_delta
        raw_deg[c] = float(d_raw)
        used_deg[c] = float(d_used)

    return raw_deg, used_deg, observed_delta


def estimate_base_lap_times(nonpit_laps, compounds):
    base = {}
    for c in compounds:
        dfc = nonpit_laps[nonpit_laps['CompoundU'] == c].copy()
        if dfc.empty:
            continue
        first_laps = dfc[dfc['TyreLap'] == 1]['LapTimeSec']
        if first_laps.empty:
            first_laps = dfc[dfc['TyreLap'] <= 2]['LapTimeSec']
        if first_laps.empty:
            first_laps = dfc['LapTimeSec']
        base[c] = float(first_laps.median())
    return base


def estimate_pit_loss_from_local_baseline(driver_laps, window=4):
    laps = driver_laps.copy()
    laps = laps[laps['LapTime'].notna()].sort_values('LapNumber').copy()
    laps['LapTimeSec'] = laps['LapTime'].dt.total_seconds()

    losses = []
    details = []

    for row in laps[laps['PitInTime'].notna()].itertuples(index=False):
        in_lap = int(row.LapNumber)
        out_lap = in_lap + 1
        out_rows = laps[laps['LapNumber'] == out_lap]
        if out_rows.empty:
            continue
        out_row = out_rows.iloc[0]

        pre_window = laps[
            (laps['LapNumber'] >= in_lap - window)
            & (laps['LapNumber'] <= in_lap - 1)
            & laps['PitInTime'].isna()
            & laps['PitOutTime'].isna()
        ]

        post_window = laps[
            (laps['LapNumber'] >= out_lap + 1)
            & (laps['LapNumber'] <= out_lap + window)
            & laps['PitInTime'].isna()
            & laps['PitOutTime'].isna()
        ]

        if pre_window.empty or post_window.empty:
            continue

        expected_pair = float(pre_window['LapTimeSec'].median() + post_window['LapTimeSec'].median())
        actual_pair = float(row.LapTimeSec + out_row['LapTimeSec'])
        loss = actual_pair - expected_pair

        if loss > 0:
            losses.append(loss)
            details.append({
                'pit_in_lap': in_lap,
                'pit_out_lap': out_lap,
                'pit_loss_seconds': loss
            })

    if not losses:
        raise ValueError('Could not estimate pit loss from local baseline windows.')

    return float(np.median(losses)), pd.DataFrame(details)


driver_nonpit = prepare_nonpit_laps(all_laps, group_cols=['Stint'])
driver_nonpit = driver_nonpit[driver_nonpit['CompoundU'].isin(COMPOUNDS_TO_ANALYZE)].copy()
if driver_nonpit.empty:
    raise ValueError('No clean non-pit laps after filtering. Try another race/driver.')

fuel_track_gain_sec_per_lap = estimate_fuel_track_gain_from_field(session.laps)
raw_deg_by_compound, deg_rate_by_compound, observed_delta_by_compound = estimate_driver_deg_rates(
    driver_nonpit,
    compounds=COMPOUNDS_TO_ANALYZE,
    fuel_track_gain=fuel_track_gain_sec_per_lap
)
if not deg_rate_by_compound:
    raise ValueError('Could not estimate degradation rates for selected compounds.')

base_lap_by_compound = estimate_base_lap_times(driver_nonpit, list(deg_rate_by_compound.keys()))
PIT_STOP_LOSS_SECONDS, pit_loss_events_df = estimate_pit_loss_from_local_baseline(all_laps)

clean_laps_by_compound = {
    c: driver_nonpit[driver_nonpit['CompoundU'] == c].copy()
    for c in deg_rate_by_compound.keys()
}

model_inputs = {
    'deg_rate_by_compound': deg_rate_by_compound,
    'pit_stop_loss_seconds': PIT_STOP_LOSS_SECONDS,
    'fuel_track_gain_sec_per_lap': fuel_track_gain_sec_per_lap
}

print('model_inputs =')
print(model_inputs)
display(pit_loss_events_df)

summary_df = pd.DataFrame({
    'compound': list(deg_rate_by_compound.keys()),
    'observed_median_lap_delta_sec': [observed_delta_by_compound[c] for c in deg_rate_by_compound],
    'raw_estimated_deg_sec_per_lap': [raw_deg_by_compound[c] for c in deg_rate_by_compound],
    'strategy_deg_used_sec_per_lap': [deg_rate_by_compound[c] for c in deg_rate_by_compound],
    'decomposition_check_d_plus_gain': [deg_rate_by_compound[c] + fuel_track_gain_sec_per_lap for c in deg_rate_by_compound],
    'T0_for_partB_sec': [base_lap_by_compound[c] for c in deg_rate_by_compound]
}).sort_values('compound')
display(summary_df)

In [ ]:
plt.figure(figsize=(8, 4))
pd.Series(deg_rate_by_compound).sort_index().plot(kind='bar')
plt.ylabel('Estimated degradation d (sec/lap)')
plt.title(f'Data-derived tyre degradation rates ({DRIVER}, {YEAR} {GRAND_PRIX})')
plt.show()

## Part B) AP model to find optimal pitting strategies

Use the extracted values from Part A in:

$$T_n = T_0 + (n-1)d$$

In [ ]:
TOTAL_LAPS = int(all_laps['LapNumber'].max())

# Strategy search settings
MAX_STOPS = 3
MIN_STINT_LAPS = 8
REQUIRE_TWO_COMPOUNDS = True

available_compounds = list(deg_rate_by_compound.keys())

def stint_time_ap(t0, d, n_laps):
    # Sum_{n=1..N} [T0 + (n-1)d] = N/2 * (2*T0 + (N-1)d)
    return n_laps / 2 * (2 * t0 + (n_laps - 1) * d)


def total_strategy_time(stint_lengths, stint_compounds, base_lap_dict, deg_dict, pit_loss):
    if len(stint_lengths) != len(stint_compounds):
        raise ValueError('stint_lengths and stint_compounds must have same length')
    driving = 0.0
    for N, c in zip(stint_lengths, stint_compounds):
        driving += stint_time_ap(base_lap_dict[c], deg_dict[c], N)
    pit_stops = len(stint_lengths) - 1
    return driving + pit_stops * pit_loss


def generate_stint_partitions(total_laps, num_stints, min_stint):
    results = []

    def backtrack(remaining, k, current):
        if k == 1:
            if remaining >= min_stint:
                results.append(current + [remaining])
            return
        max_first = remaining - min_stint * (k - 1)
        for first in range(min_stint, max_first + 1):
            backtrack(remaining - first, k - 1, current + [first])

    backtrack(total_laps, num_stints, [])
    return results


candidates = []

for stops in range(0, MAX_STOPS + 1):
    stints = stops + 1
    length_options = generate_stint_partitions(TOTAL_LAPS, stints, MIN_STINT_LAPS)

    for lengths in length_options:
        for compounds_seq in product(available_compounds, repeat=stints):
            if REQUIRE_TWO_COMPOUNDS and stints > 1 and len(set(compounds_seq)) < 2:
                continue

            t = total_strategy_time(
                stint_lengths=lengths,
                stint_compounds=compounds_seq,
                base_lap_dict=base_lap_by_compound,
                deg_dict=deg_rate_by_compound,
                pit_loss=PIT_STOP_LOSS_SECONDS
            )

            candidates.append({
                'stops': stops,
                'stint_lengths': lengths,
                'stint_compounds': compounds_seq,
                'estimated_total_time_s': t
            })

if not candidates:
    raise ValueError('No valid strategies generated. Relax constraints.')

results_df = pd.DataFrame(candidates).sort_values('estimated_total_time_s').reset_index(drop=True)
best = results_df.iloc[0]

def format_strategy(lengths, compounds):
    return ' | '.join([f'{c}-{n}' for c, n in zip(compounds, lengths)])

results_df['strategy'] = [format_strategy(l, c) for l, c in zip(results_df['stint_lengths'], results_df['stint_compounds'])]

print(f'Total race laps used: {TOTAL_LAPS}')
print(f'Best strategy: {best["stint_compounds"]} with stint lengths {best["stint_lengths"]}')
print(f'Estimated total time: {best["estimated_total_time_s"]:.2f} s')

display(results_df[['stops', 'strategy', 'estimated_total_time_s']].head(10))

### Notes

- Part A uses constrained decomposition on lap-to-lap deltas (not AP fitting).
- Part B is where AP is used to model stint lap times for strategy comparison.
- Real race strategy also depends on traffic, safety cars, and undercut/overcut effects.